In [24]:
# ── Paso 0: imports ───────────────────────────────────────────────────────────
from pyspark.sql.functions import input_file_name, regexp_extract, col, current_timestamp
from delta.tables import DeltaTable
import notebookutils

BRONZE_PATH   = "Files/bronze/gps_raw"
SOURCE_FOLDER = "Files/source/gps_csv"
PROCESADOS    = "Files/source/gps_procesados"
CONTROL_PATH  = "Files/bronze/_control_archivos"




StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 26, Finished, Available, Finished, False)

In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, trim, current_timestamp,
    input_file_name, regexp_extract
)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 28, Finished, Available, Finished, False)

In [27]:
# ── Paso 1: leer el registro de archivos ya procesados ────────────────────────
# La primera vez no existe → lo creamos vacío
try:
    df_control = spark.read.format("delta").load(CONTROL_PATH)
    ya_procesados = set(df_control.select("archivo").rdd.flatMap(lambda x: x).collect())
    print(f"Archivos ya procesados anteriormente: {len(ya_procesados)}")
except:
    ya_procesados = set()
    print("Primera ejecución — no hay control previo")

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 29, Finished, Available, Finished, False)

Primera ejecución — no hay control previo


In [28]:
# ── Paso 2: listar archivos disponibles en la zona de entrada ─────────────────
archivos_disponibles = [
    f.path for f in notebookutils.fs.ls(SOURCE_FOLDER)
    if f.name.endswith(".csv")
]

# Filtrar solo los que NO han sido procesados
archivos_nuevos = [
    f for f in archivos_disponibles
    if f.split("/")[-1] not in ya_procesados
]

print(f"Archivos disponibles:  {len(archivos_disponibles)}")
print(f"Archivos nuevos:       {len(archivos_nuevos)}")
print(f"Ya procesados (skip):  {len(ya_procesados)}")

if not archivos_nuevos:
    print("Nada nuevo que procesar — fin del notebook")
    notebookutils.notebook.exit("sin_archivos_nuevos")

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 30, Finished, Available, Finished, False)

Archivos disponibles:  4
Archivos nuevos:       4
Ya procesados (skip):  0


In [29]:
# Estructura del archivo (primeras 5 líneas):
# Línea 1: "Cuéntanos tu Experiencia INFORME PERSONALIZADO..."
# Línea 2: "Fecha Inicial:2026-02-25 Fecha Final:2026-03-31"
# Línea 3: "Sistema: gsm Celular # : 70094641 Placa: C182946"
# Línea 4: (vacía)
# Línea 5: ESTADO|FECHA|HORA|LATITUD|LONGITUD|...  ← header real
# Línea 6+: datos

# ── Paso 3: leer solo los archivos nuevos ────────────────────────────────────
df_crudo = (
    spark.read
    .format("csv")
    .option("header",   "false")
    .option("sep",      "|")
    .option("encoding", "ISO-8859-1")
    .option("mode",     "PERMISSIVE")
    .load(archivos_nuevos)          # ← solo los nuevos, no todos
)


StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 31, Finished, Available, Finished, False)

In [31]:
# ── Paso 2: filtrar solo filas de datos reales ────────────────────────────────
# _c0 = "Ok" o "Error" en datos reales
# Las 3 líneas de metadata + 1 header + vacías se descartan
df_datos = df_crudo.filter(col("_c0").isin("Ok", "Error"))

columnas = [
    "ESTADO", "FECHA", "HORA", "LATITUD", "LONGITUD", "VELOCIDAD",
    "ALTITUD", "GEOREFERENCIA", "MOTIVO", "ENTRADA_UNO",
    "BATERIA_BACKUP_MV", "BATERIA_PRINCIPAL_MV", "RUMBO",
    "DISTANCIA_RECORRIDA", "FECHA_GRABACION", "GPS", "TEMPERATURA"
]
for i, nombre in enumerate(columnas):
    df_datos = df_datos.withColumnRenamed(f"_c{i}", nombre)

df_datos = (
    df_datos
    .drop("_c17")
    .withColumn("_source_file", input_file_name())
    .withColumn("PLACA", regexp_extract(col("_source_file"), r"(C\d{6})", 1))
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_nombre_archivo",                          # guardamos el nombre para el control
        regexp_extract(col("_source_file"), r"([^/]+\.csv)$", 1))
    .drop("_source_file")
)

print(f"Registros leídos: {df_datos.count():,}")
df_datos.groupBy("PLACA").count().show()


#display(df_datos.select("PLACA", "FECHA", "HORA", "LATITUD", "LONGITUD", "VELOCIDAD", "MOTIVO"))

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 33, Finished, Available, Finished, False)

Registros leídos: 137,754
+-------+-----+
|  PLACA|count|
+-------+-----+
|C202944|53422|
|C202013|33476|
|C202946|26553|
|C202945|24303|
+-------+-----+



In [34]:
display(df_datos)

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dc302704-d46c-4612-b154-a29582573639)

In [35]:
# ── Paso 4: escribir a Bronze (append — nunca overwrite) ──────────────────────
(
    df_datos
    .drop("_nombre_archivo")        # no va al Bronze, es solo para el control
    .write
    .format("delta")
    .mode("append")                 # ← append, no overwrite
    .option("mergeSchema", "true")
    .partitionBy("PLACA", "FECHA")
    .save(BRONZE_PATH)
)

print("Datos agregados a Bronze")

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 37, Finished, Available, Finished, False)

Datos agregados a Bronze


In [37]:
# ── Paso 5 corregido: registrar archivos procesados ───────────────────────────

# Extraer nombres únicos DIRECTAMENTE de la lista que ya procesamos
# (más confiable que intentar recuperarlos del DataFrame)
from pyspark.sql import Row
from datetime import datetime

nombres_procesados = [ruta.split("/")[-1] for ruta in archivos_nuevos]
ts_ahora = datetime.now()

rows = [Row(archivo=nombre, fecha_procesado=ts_ahora) for nombre in nombres_procesados]
df_control_nuevo = spark.createDataFrame(rows)

(
    df_control_nuevo
    .write
    .format("delta")
    .mode("append")
    .save(CONTROL_PATH)
)

print("Control actualizado:")
spark.read.format("delta").load(CONTROL_PATH).orderBy("fecha_procesado").show(truncate=False)

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 39, Finished, Available, Finished, False)

Control actualizado:
+--------------------------------------------------+--------------------------+
|archivo                                           |fecha_procesado           |
+--------------------------------------------------+--------------------------+
|                                                  |2026-04-13 21:58:34.934872|
|C202945_SelectTOA_Personalizado_20260413115601.csv|2026-04-13 21:59:37.888243|
|C202946_SelectTOA_Personalizado_20260413115652.csv|2026-04-13 21:59:37.888243|
|C202944_SelectTOA_Personalizado_20260413114811.csv|2026-04-13 21:59:37.888243|
|C202013_SelectTOA_Personalizado_20260413114548.csv|2026-04-13 21:59:37.888243|
+--------------------------------------------------+--------------------------+



In [41]:
# ── Paso 6: mover archivos procesados ────────────────────────────────────────

# Crear la carpeta destino si no existe
try:
    notebookutils.fs.ls(PROCESADOS)
except:
    notebookutils.fs.mkdirs(PROCESADOS)
    print(f"Carpeta creada: {PROCESADOS}")

# Mover cada archivo
for ruta_completa in archivos_nuevos:
    nombre = ruta_completa.split("/")[-1]
    destino = f"{PROCESADOS}/{nombre}"
    try:
        notebookutils.fs.mv(ruta_completa, destino, overwrite=True)
        print(f"  Movido: {nombre}")
    except Exception as e:
        print(f"  Error moviendo {nombre}: {e}")

print("\n Notebook 01 completo")

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 43, Finished, Available, Finished, False)

  Error moviendo C202013_SelectTOA_Personalizado_20260413114548.csv: An error occurred while calling z:notebookutils.fs.mv.
: java.io.FileNotFoundException: abfss://557aefe8-387c-4a7d-93ea-752bae49db96@onelake.dfs.fabric.microsoft.com/aacf8392-a280-483c-95af-9e6b624a6608/Files/source/gps_csv/C202013_SelectTOA_Personalizado_20260413114548.csv not found
	at com.microsoft.spark.notebook.msutils.impl.MSFsUtilsImpl.$anonfun$mv$2(MSFsUtilsImpl.scala:345)
	at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.java:23)
	at com.microsoft.spark.notebook.msutils.impl.MSFsUtilsImpl.fsTSG(MSFsUtilsImpl.scala:223)
	at com.microsoft.spark.notebook.msutils.impl.MSFsUtilsImpl.$anonfun$mv$1(MSFsUtilsImpl.scala:332)
	at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.java:23)
	at com.microsoft.spark.notebook.common.trident.CertifiedTelemetryUtils$.withTelemetry(CertifiedTelemetryUtils.scala:98)
	at com.microsoft.spark.notebook.msutils.impl.MSFsUtilsImpl.mv(MSFsUtilsImpl.scala:332

In [42]:
# ── Paso 6: verificación ──────────────────────────────────────────────────────
df_check = spark.read.format("delta").load("Files/bronze/gps_raw")

print(f"Total registros: {df_check.count():,}")
print()

# Registros por placa
df_check.groupBy("PLACA").count().orderBy("PLACA").show()

# Rango de fechas por placa
df_check.groupBy("PLACA").agg(
    {"FECHA": "min", "FECHA": "max"}
).show()

# Muestra de datos
display(df_check.select(
    "PLACA", "FECHA", "HORA", "LATITUD", "LONGITUD", "VELOCIDAD", "MOTIVO"
).limit(10))

StatementMeta(, 60efe5cb-2780-4ec5-88cf-0ad6e9822241, 44, Finished, Available, Finished, False)

Total registros: 137,754

+-------+-----+
|  PLACA|count|
+-------+-----+
|C202013|33476|
|C202944|53422|
|C202945|24303|
|C202946|26553|
+-------+-----+

+-------+----------+
|  PLACA|max(FECHA)|
+-------+----------+
|C202013|2026-04-01|
|C202944|2026-03-31|
|C202945|2026-03-31|
|C202946|2026-03-31|
+-------+----------+



SynapseWidget(Synapse.DataFrame, f4e8258d-5dba-40dc-914d-4ed80963c411)